Part 0

The PragmatiCQA dataset

Key motivations and contributions:

1. When using LLM's as a researching tool, having a question answered literally isn't particulary helpful. A human answer typically provides more information and context besides what was asked by inferring pragmatical information from the asker. The dataset provides human generated conversational question-answer pairs where the pragmatic answers have additional information and context beyond what was asked, anticipating follow up questions and being generally cooperative.

2. Existing datasets exploring CQA evaluated answers based on factual correctness and not cooperative pragmatic reasoning. Additionally, the teachers weren't insentivised to answer cooperatively, only to complete their task - which is answer the question and not necessarily create a welcoming learning experience. The PragmatiCQA mitigates this in several ways. First, teachers are incentivised to keep the conversations longer, and the students rate the teacher's answers afterwards. The students are incentivised to learn from their teacher's answers. Both students and teachers chose the topic they're interested in and were paired accordingly.

3. The PragmatiCQA Dataset provides diverse topics (73) based on community fandoms spanning comics, movies, video games and music, so evaluations and training can be done better.

4. The metrics used for evaluation take into consideration the answer's accuracy, pragmatic reasoning, naturalness and faithfulness.


What makes this dataset challenging for NLP models?
What specific pragmatic phenomena does it target?

The task in more than retrieving an answer to a query, but to make a cooperative conversation.

It means answering as  human-like as possible, therefore inferring unspoken interests the student might have, and answering them.

Pragmatical interest might also be related to information only accessible to the teacher, like identifying facts that make this question interesting from the database.

Having a flowing conversation means not repeating information already discussed, but keeping it in mind when answering a new question.

Pitfalls for LLMs might be hallucinations, Entity ambiguity (Batman can be a comic, a TV show, a movie, animation, etc.) and reduntant over supplying of information.

Examples:

1. Student: Are there any lego sets that represent barbie?
    
    This is literally a yes/no question, but it implies a lot about the student's interest.

    The answer 'No' can be enriched with information about other LEGO sets similar to Barbie, or suggestions about recreating Barbie scenes using LEGO.

2.  Student: Who is snoopy's fiancee? I didn't know he had one.

    This is a difficult question to answer, since she is never given a name, therefore hard identify as an entity.

    Adding information about where she appears, how she is reffered to, what are her attributes could enrich the answer.

    In general, any 'Who is x' question can be enriched beyond its literal meaning in many ways.

3.  Student: Do they review all types of movies? (in the context of MST3K)

    The show riffs on bottom-of-the-barrel B-movies, mostly Sci-Fi. 
    
    This can be enriched with why this format works for these type of movies

    Give an example of a movie that producesed some really memorable jokes.

4.  Student: Who is the king at the end of the book? (in the context of Throne of Glass)

    Answering this could be considered a spoiler, but disregarding that.

    Enriching the answer with the qualities of said ruler - were they just? Were they worthy? Is it considered a good ending that they took the throne?

5.  Student: Who is the protagonist? (in the context of Fallout)

    There is ambiguity here, since there are different protagonists in each game, and there is also a TV show.

    The answer can be enriched with the protagonist's back story (or maybe lack-there-of), their goals, their characteristics, their defining moments in the narrative, etc. 

Part 1

The "Traditional" NLP Approach

The model used:

In [2]:
from transformers import pipeline
question_answerer = pipeline("question-answering", model='distilbert-base-cased-distilled-squad')

Device set to use cpu


The evaluation method, the retrieval method and the database:

In [3]:
import dspy
from sentence_transformers import SentenceTransformer

# Load an extremely efficient local model for retrieval
model = SentenceTransformer("sentence-transformers/static-retrieval-mrl-en-v1", device="cpu")

# Create an embedder using the model's encode method
embedder = dspy.Embedder(model.encode)

# Traverse a directory and read html files - extract text from the html files
import os
from bs4 import BeautifulSoup
def read_html_files(directory):
    texts = []
    for filename in os.listdir(directory):
        if filename.endswith(".html"):
            with open(os.path.join(directory, filename), 'r', encoding='utf-8') as file:
                soup = BeautifulSoup(file, 'html.parser')
                texts.append(soup.get_text())
    return texts

The limit is 3 since I'm reaching a token limit pretty consistently

In [4]:
# Parameters for the retriever
max_characters = 10000  # for truncating >99th percentile of documents
topk_docs_to_retrieve = 3  # number of documents to retrieve per search query


In [ ]:
import json
import os  

def read_data(filename, dataset_dir="../PragmatiCQA/data"):
    corpus = []
    with open(os.path.join(dataset_dir, filename), 'r') as f:
        for line in f:
            corpus.append(json.loads(line))
    return corpus

pcqa_test = read_data("test.jsonl")

In [133]:
lm = dspy.LM('xai/grok-3-mini', max_tokens=6000, temperature=0.1, top_p=0.9)
dspy.configure(lm=lm)
with open("grok_key.ini") as f:
    for line in f:
        if "XAI_API_KEY" in line and not line.strip().startswith("#"):
            key_value = line.strip().split("=")
            if len(key_value) == 2:
                os.environ["XAI_API_KEY"] = key_value[1].split()[0]

In [7]:
from dspy.evaluate import SemanticF1

# Instantiate the metric.
metric = SemanticF1(decompositional=True)

In [8]:
import pprint   
pprint.pprint(pcqa_test[0]['qas'][0])

{'a': 'The Legend of Zelda came out as early as 1986 for the Famicom in Japan, '
      'and was later released in the western world, including Europe and the '
      'US in 1987. Would you like to know about the story?',
 'a_meta': {'literal_obj': [{'endKey': '9cbccabd-66be-4a46-bd8b-f59a299c987d',
                             'startKey': '1f4f808a-8560-4894-b892-15fa3c33887a',
                             'text': 'FDS release February 21, 1986\n'},
                            {'endKey': '738bff65-b4f9-4660-bd18-79722ed67a40',
                             'startKey': 'a0d9d5c5-18bb-4be4-825e-fca2900db18e',
                             'text': 'The Legend of Zelda is the first '
                                     'installment of the Zelda series. '},
                            {'endKey': '738bff65-b4f9-4660-bd18-79722ed67a40',
                             'startKey': '738bff65-b4f9-4660-bd18-79722ed67a40',
                             'text': ' It centers its plot around a boy named 

Pre-loading the topic retrieval objects

In [14]:
topic_search = {}


for item in pcqa_test:
    community = item['community']
    if community not in topic_search:
        topic_search[community] = dspy.retrievers.Embeddings(embedder=embedder, corpus=read_html_files(f"../PragmatiCQA-sources/{community}"), k=topk_docs_to_retrieve)

Running all 3 tests

In [52]:
literal_references = []
literal_predictions = []
pragmatic_references = []
pragmatic_predictions = []
rag_references = []
rag_predictions = []

for item in pcqa_test:
    topic = item['topic']
    gernre = item['genre']
    community = item['community']
    qas = item['qas'][0]
    question = qas['q']
    answer = qas['a']
    lit_spans = [l['text'] for l in qas['a_meta']['literal_obj']]
    lit_answer = ' '.join(lit_spans)
    prag_spans = [l['text'] for l in qas['a_meta']['pragmatic_obj']]
    prag_answer = ' '.join(prag_spans)
    rag_context = topic_search[community](question)

    print(question)
    
    result_literal = question_answerer(question=question,     context=lit_answer)
    literal_references.append(dspy.Example(question=question, response=answer, inputs={'context': lit_answer}))
    literal_predictions.append(dspy.Prediction(question = question, response = result_literal['answer']).with_inputs(lit_answer))
    print(f"Answer with literal context: '{result_literal['answer']}'.")
    
    result_pragmatic = question_answerer(question=question,     context=prag_answer)
    pragmatic_references.append(dspy.Example(question=question, response=answer, inputs={'context': prag_answer}))
    pragmatic_predictions.append(dspy.Prediction(question = question, response = result_pragmatic['answer']).with_inputs(prag_answer))
    print(f"Answer with pragmatic context: '{result_pragmatic['answer']}'.")
    
    result_rag = question_answerer(question=question,     context=" ".join(rag_context.passages))
    rag_references.append(dspy.Example(question=question, response=answer, inputs={'context': rag_context}))
    rag_predictions.append(dspy.Prediction(question = question, response = result_rag['answer']).with_inputs(" ".join(rag_context.passages)))
    print(f"Answer with rag context: '{result_rag['answer']}'.")


What year did the Legend of Zelda come out?
Answer with literal context: '1986'.
Answer with pragmatic context: '1986'.
Answer with rag context: '1986'.
What console is The Legend of Zelda designed for?
Answer with literal context: 'Famicom'.
Answer with pragmatic context: 'Nintendo Entertainment System'.
Answer with rag context: 'Game Boy Color'.
when did the legend of zelda last until?
Answer with literal context: 'first installment in the Zelda franchise'.
Answer with pragmatic context: 'April 23, 2019'.
Answer with rag context: 'June 19, 2011'.
When was the Legend of Zelda released?
Answer with literal context: 'August 22, 1987'.
Answer with pragmatic context: '1987'.
Answer with rag context: '1986'.
What kind of game is The Legend of Zelda?
Answer with literal context: 'Zelda'.
Answer with pragmatic context: 'roleplaying'.
Answer with rag context: 'multiplayer'.
What year was this game release?
Answer with literal context: '1986'.
Answer with pragmatic context: '1987'.
Answer with

Evaluating

In [ ]:

literal_examples = []
for pred, reference in zip(literal_predictions, literal_references):
    literal_examples.append(dspy.Example(example = reference, pred = pred).with_inputs("example" , "pred"))

pragmatic_examples = []
for pred, reference in zip(pragmatic_predictions, pragmatic_references):
    pragmatic_examples.append(dspy.Example(example = reference, pred = pred).with_inputs("example" , "pred"))

rag_examples = []
for pred, reference in zip(rag_predictions, rag_references ):
    rag_examples.append(dspy.Example(example = reference, pred = pred).with_inputs("example" , "pred"))

literal_score = metric.batch(literal_examples)
pragmatic_score = metric.batch(pragmatic_examples)
rag_score = metric.batch(rag_examples)

Processed 213 / 213 examples: 100%|██████████| 213/213 [00:09<00:00, 23.41it/s]  


In [56]:
#evaluate

print(f"Mean results with literal context: {sum(literal_score) / 213}")
print(f"Mean results with pragmatic context: {sum(pragmatic_score) / 213}")
print(f"Mean results with rag context: {sum(rag_score) / 213}")

Mean results with literal context: 0.4088523710545823
Mean results with pragmatic context: 0.3634055940982089
Mean results with rag context: 0.10846077147389875


Evaluation:

From the score number, we can clearly see the model had trouble providing a relevant answer using the retrieved context.
Scores overall weren't high either. 

Looking at some of the answers, all were short, some even had just a single token (like a year or a name).

The model attempted at giving a literal answer, whether true or not, without pragmatic inferrence and without providing additional context to the answer.

It also never provided potential follow-up questions, and didn't answer conversationally.

Also, this is just funny:

Who is Snoopy?

Answer with literal context: 'a dog'.

Answer with pragmatic context: 'loves root beer and pizza'.

Answer with rag context: 'Tarzan'.

In [58]:
cost = sum([x['cost'] for x in lm.history if x['cost'] is not None])  # in USD, as calculated by LiteLLM for certain providers
print(cost)

0.8963926000000042


Part 2

Creating a complex LLM model

Two options, a multi-hop dspy model, which will compute step-by-step a cooperative response,

or a dspy ReAct agent, which will use search tools and inference tools as it sees fit.

The response should be:

1. Not copy-pasted, conversational.

2. Answers the question without providing excess information.

3. Goes beyond literal answer and anticipates would-be followup questions and provides relevant context.

4. Provides helpful leads to help ask further questions on this topic.

5. Faithfully represents information retrieved by the queries.

The process should involve:

1. Retrieve context from the community database using the question as query.

2. Generate a summary of the student's goals or interests based on the conversation history.

3. Generate a pragmatic or cooperative need underlying the student's current question based on past conversation and retrieved context.

4. Generate a cooperative question to re-query the source documents and extract additional context.

5. Compose a short conversational response based on generated infromation and built as indicated.



MultiHop model:

Model 1: query, rag data -> answer_spans [RAG model]

Model 2:question, conversation history, community -> literal query, pragmatic query [pragmatic model]

model 3: question, conversation history, literal answer, pragmatic answer -> conversational response with added follow-up suggestion [response model]

Model 4: question, conversation history, community -> model 3 output. [cooperative teacher model]

Flow: 

user question + conversation history + community fed to pragmatic model.

queries from pragmatic model are both fed to RAG model.

question + conversation history + literal and pragmatic answers fed to response model

return response model prediction.

First - RAG model.

Goal - create a concise list of copy-pasted facts that answer a query.

In [59]:
class RAG_signature(dspy.Signature):

    """Using the provided `context` and `question`, provide a list of sentences copied from the context that answer the question."""
    
    context: str = dspy.InputField() 
    question: str = dspy.InputField()
    answer_list: list[str] = dspy.OutputField()
    
RAG_prompt = dspy.ChainOfThought(RAG_signature)

context=" ".join(topic_search["The Legend of Zelda"]("What year did the Legend of Zelda come out?").passages)
question = "What year did the Legend of Zelda come out?"

print(RAG_prompt(context = context, question = question))

Prediction(
    reasoning='The question asks for the year The Legend of Zelda came out. In the provided context, there are specific sentences that mention the release year of the game. I identified sentences that directly reference the initial release dates, focusing on the original Japanese release in 1986, as it is the earliest mentioned. These sentences are copied verbatim from the context to ensure accuracy.',
    answer_list=['It came out as early as 1986 for the Famicom in Japan, and was later released in the western world, including Europe and the US in 1987.', 'The Legend of Zelda was originally released in 1986 as a flagship title for the Famicom Disk System in Japan.']
)


Second - Pragmatics model

This one was tricky, but results were improved when I made the decision to produce both the literal query and the pragmatic query at once.

Also produce an 'interest' to help direct the queries.

Producing both queries at once helped in forcing the LLM into creating a query that doesn;t answer the literal question.

In [62]:
class pragmatics_signature(dspy.Signature):

    """Using the provided 'conversation_history' between a student and a teacher, your role is to derive a pragmatic or cooperative 'interest' underlying the student's current question beyond its literal meaning provided in the field 'question' and to generate two one sentence queries phrased as questions for the database, one about the student's literal question and one for additional information based on the pragmatically derived 'interest'."""
    
    conversation_history: list[str] = dspy.InputField() 
    question: str = dspy.InputField()
    interest: str = dspy.OutputField()
    literal_query: str = dspy.OutputField()
    pragmatic_query: str = dspy.OutputField()
    
pragmatics_prompt = dspy.ChainOfThought(pragmatics_signature)

conversation_history_1 = ["Student: What year did the Legend of Zelda come out?"]
conversation_history_2 = ["Student: What year did the Legend of Zelda come out?",
                          "Teacher: The Legend of Zelda came out as early as 1986 for the Famicom in Japan, and was later released in the western world, including Europe and the US in 1987. Would you like to know about the story?",
                          "Student: Yes who are the main characters in the story?"]
question_2 = "Yes who are the main characters in the story?"

print(pragmatics_prompt(conversation_history = conversation_history_1, question = question))
print(pragmatics_prompt(conversation_history = conversation_history_2, question = question_2))

Prediction(
    reasoning="The student's question directly asks for the release year of the Legend of Zelda, which is a straightforward factual inquiry, but pragmatically, this could indicate a broader interest in the game's historical context, such as its development, cultural impact, or place in the evolution of video games, as people often ask for release dates to build timelines or understand a series' origins.",
    interest='Interest in the historical context and cultural significance of the Legend of Zelda series.',
    literal_query='What is the release year of the Legend of Zelda?',
    pragmatic_query='What are some key historical facts or milestones in the development and impact of the Legend of Zelda series?'
)
Prediction(
    reasoning="Based on the conversation history, the student initially asked about the release year of The Legend of Zelda, and after the teacher's response and suggestion to learn about the story, the student affirmed interest and specifically asked abo

Third model - response model

I wanted to produce an answer from all information gathered and wrap it in a nice conversational answer.

This is the model which will be tested and optimized.

In [107]:
class conversational_response_signature(dspy.Signature):

    """Using the provided 'conversation_history' between a student and a teacher, and a student's question in the field 'question', your role is to answer as the teacher in a conversational manner using the information provided in the 'literal_answers' and the 'pragmatic_answers' fields. Your response should be natural-sounding, faithful to the information provided, and end with a leading question to the student to help them keep the conversation flowing."""
    conversation_history: list[str] = dspy.InputField() 
    question: str = dspy.InputField()
    literal_answers: list[str] = dspy.InputField()
    pragmatic_answers: list[str] = dspy.InputField()
 #   literal_key_information: str = dspy.OutputField()
 #   pragmatic_key_information: str = dspy.OutputField()
#    follow_up_question: str = dspy.OutputField()
    response: str = dspy.OutputField()
    
conversational_response_prompt = dspy.ChainOfThought(conversational_response_signature)


Manually  testing one input:

In [ ]:

step_1 = pragmatics_prompt(conversation_history = conversation_history_1, question = question)
print(step_1)
search_1 = topic_search['The Legend of Zelda'](step_1.literal_query)
search_2 = topic_search['The Legend of Zelda'](step_1.pragmatic_query)
step_2_1 = RAG_prompt(context = " ".join(search_1.passages), question = question)
print(step_2_1)
step_2_2 = RAG_prompt(context = " ".join(search_2.passages), question = step_1.pragmatic_query)
print(step_2_2)
step_3 = conversational_response_prompt(conversation_history = conversation_history_1, question = question, literal_answers = step_2_1.answer_list, pragmatic_answers = step_2_2.answer_list)
print(step_3)

In [67]:
metric(dspy.Example(
    question = question, 
    literal_answers = [l['text'] for l in pcqa_test[0]['qas'][0]['a_meta']['literal_obj']],
    pragmatic_answers = [l['text'] for l in pcqa_test[0]['qas'][0]['a_meta']['pragmatic_obj']],
    response = "The Legend of Zelda came out as early as 1986 for the Famicom in Japan, and was later released in the western world, including Europe and the US in 1987. Would you like to know about the story?").with_inputs("question", "literal_answers", "pragmatic_answers"), 
    pred = dspy.Prediction(
        question = question, 
        literal_answers = step_2_1.answer_list,
        pragmatic_answers = step_2_2.answer_list,
        response = step_3.conversational_response).with_inputs("question", "literal_answers", "pragmatic_answers"))

0.4444444444444444

Not a very good score. Let's test the response with data from the dataset instead of my models.

In [69]:
step_3_with_dataset_spans = conversational_response_prompt(
    conversation_history = conversation_history_1, 
    question = question, 
    literal_answers = [l['text'] for l in pcqa_test[0]['qas'][0]['a_meta']['literal_obj']], 
    pragmatic_answers = [l['text'] for l in pcqa_test[0]['qas'][0]['a_meta']['pragmatic_obj']]
)
print(step_3_with_dataset_spans)
metric(dspy.Example(
    question = question, 
    response = "The Legend of Zelda came out as early as 1986 for the Famicom in Japan, and was later released in the western world, including Europe and the US in 1987. Would you like to know about the story?").with_inputs("question"), 
    pred = dspy.Prediction(
        question = question, 
        response = step_3_with_dataset_spans.conversational_response).with_inputs("question"))

Prediction(
    reasoning="The student's question is about the release year of The Legend of Zelda, which matches the initial conversation history. The literal_answers provide specific details like the release date (February 21, 1986) and context about the game being the first in the series with Link as the protagonist, but these are fragmented and technical. The pragmatic_answers offer a more conversational and comprehensive summary, noting the 1986 release in Japan and 1987 in the West, which is ideal for a natural-sounding teacher response. I'll base my answer on the pragmatic information to keep it accurate and engaging, while briefly incorporating relevant literal details for completeness. The response will be conversational, start with a clear answer, add a bit of educational context, and end with a leading question to encourage further discussion, such as asking about the student's interest in the game.",
    conversational_response="Well, the Legend of Zelda was first released 

0.4444444444444444

The response was different but the score is the same.

Is that a good sign or a bad one?

Maybe I should try optimizing the response separately using the dataset.

My thinking:

Optimizing the final response based on dataset inputs will help alleviate rerieval accuracy and factual correctness, based on SemanticF1 score when compared to the actual response.

Since the teachers' responses in the dataset are given a final human evaluation that is useful to me, like consistency with data spans, not copy pasted, not only literal, adds a good leading question..

Another key issue is the pragmatic model, which will sometimes be inconsistent with the teachers pragmaticly retrieved spans. However, since this doesn't necessarily mean the response is bad (there are many valid guesses), I don't want it to affect my evaluation during optimization. Therefore the pragmatics model and the retrieval model will not be part of this (they can be optimized by themselves with the dataset, but I see this a bit redundant and time/cost inefficient).

The second solution is to optimized the whole model as one. I think it would be less efficient because:

- pragmatic query might be different and would deeply affect the score while not being necessarily wrong.

- it would be very time/cost consuming, which can be mitigated by using less examples and therefore can lead to it being less optimized.


In [ ]:
pragmatics_prompt(conversation_history = "Student: Is there water on Mars?", question = "Is there water on Mars?")

Prediction(
    reasoning='The student\'s question "Is there water on Mars?" is repeated in the conversation history, suggesting it might be a standalone inquiry; however, pragmatically, this could stem from a broader interest in planetary science, such as the potential for human exploration or the existence of extraterrestrial life, as questions about water on Mars often relate to habitability and NASA\'s missions rather than just a factual yes/no answer.',
    interest='The student is interested in the implications of water on Mars for potential human colonization or the possibility of extraterrestrial life.',
    literal_query='What evidence exists regarding the presence of water on Mars?',
    pragmatic_query='What are the implications of water on Mars for the potential for extraterrestrial life or human exploration?'
)

Let's try optimizing just the 3rd model using literal and pragmatic spans from the dataset.

This way, even though the 2nd model might infer a different query than the teacher, that won't affect the quality of the response when optimizing.

First, let's currate a good training dataset

In [83]:

pcqa_train = read_data("train.jsonl")

pprint.pprint(pcqa_train[0]['qas'][0])

{'a': 'Born in California on October 20, 1971, Snoop Dog is now 46 years old',
 'a_meta': {'literal_obj': [{'endKey': 'cd2d0029-2e41-4a31-980d-01e6b4e52628',
                             'startKey': 'd8676daa-9f98-430c-9fe7-69b63b81ee93',
                             'text': 'Age\n46'}],
            'pragmatic_obj': [{'endKey': 'e4b81116-7bdb-469c-a078-12432bb607c5',
                               'startKey': 'e4b81116-7bdb-469c-a078-12432bb607c5',
                               'text': 'born October 20, 1971'},
                              {'endKey': '673d66dd-e5d4-48c5-8111-2697e5bdc4de',
                               'startKey': '673d66dd-e5d4-48c5-8111-2697e5bdc4de',
                               'text': 'in\xa0Long Beach, California'}]},
 'human_eval': ['1', '1', '1', '1', '1'],
 'q': 'how old is snoop dogg?'}


I'd like to train using answers that were better evaluated 

In [ ]:
training_corpus = []
training_list = []

for item in pcqa_train:
    ch = []
    for pair in item['qas']:
        a = pair['a']
        q = pair['q']
        llist = [ans['text'] for ans in pair['a_meta']['literal_obj']]
        plist = [ans['text'] for ans in pair['a_meta']['pragmatic_obj']]
        ch.append(f"Student: {q}")

        if "human_eval" in pair:
            avg = 0
            for score in pair['human_eval']:
                avg+=int(score)
            avg/=5
            if avg >=4:
                current_ch = ch[:]
                training_corpus.append(dspy.Example(
                    conversation_history = current_ch, 
                    question = q, 
                    literal_answers = llist,
                    pragmatic_answers = plist,
                    response = a
                ).with_inputs("conversation_history", "question", "literal_answers", "pragmatic_answers")
                )
                #i might use this later, for testing
                training_list.append({
                    "topic": item['topic'],
                    "community": item['community'],
                    "conversation_history": current_ch,
                    "literal_answers": llist,
                    "pragmatic_answers": plist,
                    "question:": q,
                    "answer": a,
                    "score": pair['human_eval']
                    
                })
        ch.append(f"Teacher: {a}")


In [ ]:
print(len(training_corpus))
pprint.pprint(training_corpus[0])

import random

training_corpus_sample = random.sample(training_corpus, 50)

121
Example({'conversation_history': ['Student: Who is Snoop dog?', 'Teacher: An American rapper who began his music career in 1992', 'Student: Was he an East Coast or West Coast please?', 'Teacher: he was born in long beach California on October 20th 1971', 'Student: So a west coast rapper. Did he have any fellow rappers he liked?', 'Teacher: he once dedicated Real talk albulm to Tookie ( former crisp leader stanley williams, and he was found by Dr dre so am guessing those guys were people he liked.', 'Student: Have you any more information about Dr Dre?', 'Teacher: Dr Dre featured Snoop on his solo debut deep cover and also the album The chronic.', 'Student: How many albums has snoop made?', 'Teacher: He is said to have sold over 23 million albums in the US and 35 million worldwide.', 'Student: Cool, what else has he done?'], 'question': 'Cool, what else has he done?', 'literal_answers': ["Snoop's debut album, Doggystyle , produced by Dr. Dre, was released in 1993 by\xa0Death Row Rec

In [115]:
from dspy.teleprompt import MIPROv2

def semanticF1_metric(example, prediction, trace=None):
    score = metric(example, pred = prediction)
    print(score)
    return score


teleprompter = MIPROv2(
    metric=semanticF1_metric,
    auto="medium", # Can choose between light, medium, and heavy optimization runs
)

optimized_response = teleprompter.compile(
    conversational_response_prompt,
    trainset = training_corpus_sample,
)


2025/08/23 13:32:17 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING MEDIUM AUTO RUN SETTINGS:
num_trials: 18
minibatch: False
num_fewshot_candidates: 12
num_instruct_candidates: 6
valset size: 40



Projected Language Model (LM) Calls

Based on the parameters you have set, the maximum number of LM calls is projected as follows:

- Prompt Generation: 10 data summarizer calls + 6 * 1 lm calls in program + (2) lm calls in program-aware proposer = 18 prompt model calls
- Program Evaluation: 40 examples in val set * 18 batches = 720 LM program calls

Estimated Cost Calculation:

Total Cost = (Number of calls to task model * (Avg Input Token Length per Call * Task Model Price per Input Token + Avg Output Token Length per Call * Task Model Price per Output Token)
            + (Number of program calls * (Avg Input Token Length per Call * Task Prompt Price per Input Token + Avg Output Token Length per Call * Prompt Model Price per Output Token).

For a preliminary estimate of potential costs, we recommend you perform your own calculations based on the task
and prompt models you intend to use. If the projected costs exceed your budget or expectations, you may consider:

- Reducing the numb

2025/08/23 13:32:37 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/08/23 13:32:37 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/08/23 13:32:37 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=12 sets of demonstrations...



No input received within 20 seconds. Proceeding with execution...
Bootstrapping set 1/12
Bootstrapping set 2/12
Bootstrapping set 3/12


 10%|█         | 1/10 [00:20<03:06, 20.72s/it]

...

2025/08/23 14:17:29 INFO dspy.evaluate.evaluate: Average Metric: 20.848990889211567 / 40 (52.1%)
2025/08/23 14:17:29 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 52.12 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 10'].
2025/08/23 14:17:29 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [51.66, 49.84, 50.69, 52.28, 51.51, 51.52, 51.94, 43.71, 52.55, 54.33, 54.33, 54.33, 50.41, 51.19, 51.03, 51.15, 52.12]
2025/08/23 14:17:29 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 54.33
2025/08/23 14:17:29 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2025/08/23 14:17:29 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 18 / 18 =====



Average Metric: 21.67 / 40 (54.2%): 100%|██████████| 40/40 [02:17<00:00,  3.44s/it]

2025/08/23 14:19:46 INFO dspy.evaluate.evaluate: Average Metric: 21.668993623156126 / 40 (54.2%)
2025/08/23 14:19:46 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 54.17 with parameters ['Predictor 0: Instruction 3', 'Predictor 0: Few-Shot Set 7'].
2025/08/23 14:19:46 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [51.66, 49.84, 50.69, 52.28, 51.51, 51.52, 51.94, 43.71, 52.55, 54.33, 54.33, 54.33, 50.41, 51.19, 51.03, 51.15, 52.12, 54.17]
2025/08/23 14:19:46 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 54.33
2025/08/23 14:19:46 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2025/08/23 14:19:46 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 19 / 18 =====



Average Metric: 20.90 / 40 (52.2%): 100%|██████████| 40/40 [02:06<00:00,  3.16s/it]

2025/08/23 14:21:53 INFO dspy.evaluate.evaluate: Average Metric: 20.899855450994202 / 40 (52.2%)
2025/08/23 14:21:53 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 52.25 with parameters ['Predictor 0: Instruction 3', 'Predictor 0: Few-Shot Set 11'].
2025/08/23 14:21:53 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [51.66, 49.84, 50.69, 52.28, 51.51, 51.52, 51.94, 43.71, 52.55, 54.33, 54.33, 54.33, 50.41, 51.19, 51.03, 51.15, 52.12, 54.17, 52.25]
2025/08/23 14:21:53 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 54.33
2025/08/23 14:21:53 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2025/08/23 14:21:53 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 54.33!


In [132]:
step_3_with_dataset_spans_and_optimized = optimized_response(
    conversation_history = conversation_history_1, 
    question = question, 
    literal_answers = [l['text'] for l in pcqa_test[0]['qas'][0]['a_meta']['literal_obj']], 
    pragmatic_answers = [l['text'] for l in pcqa_test[0]['qas'][0]['a_meta']['pragmatic_obj']]
)
print(step_3_with_dataset_spans_and_optimized)
metric(dspy.Example(
    question = question, 
    response = "The Legend of Zelda came out as early as 1986 for the Famicom in Japan, and was later released in the western world, including Europe and the US in 1987. Would you like to know about the story?").with_inputs("question"), 
    pred = dspy.Prediction(
        question = question, 
        response = step_3_with_dataset_spans_and_optimized.response).with_inputs("question"))

Prediction(
    reasoning="The student's question is about the release year of The Legend of Zelda, which is a direct follow-up from the conversation history where this is the initial query. The literal_answers provide key facts: the Famicom Disk System (FDS) release on February 21, 1986, along with context that it's the first installment of the series and focuses on the protagonist Link. These are straightforward and factual, so I'll use them as the core of the response. The pragmatic_answers add depth by explaining the regional differences, noting the 1986 release in Japan and the 1987 release in the West, which makes the answer more engaging and practical. In my reasoning, I'll ensure the response stays accurate to the provided information without speculating. To make it natural and human-like, I'll aim for an informal, enthusiastic tone with conversational flair, like using contractions or light excitement, while keeping it concise. Finally, the response will end with an open-ended

0.6666666666666666

That's a higher score!

Now for the complete Multi-Hop module:

In [144]:
class conversational_model(dspy.Module):
    community_data = {}
    
    embedder = dspy.Embedder(SentenceTransformer("sentence-transformers/static-retrieval-mrl-en-v1", device="cpu").encode)
    def __init__(self, rag_model, pragmatic_model, response_model, num_docs):
        self.rag_model = rag_model
        self.pragmatic_model = pragmatic_model
        self.response_model = response_model
        self.num_docs = num_docs

    def forward(self, question, conversation_history, community):
        if community not in self.community_data:
            self.community_data[community] = dspy.retrievers.Embeddings(embedder=self.embedder, corpus=read_html_files(f"../PragmatiCQA-sources/{community}"), k=self.num_docs)
        query_pred = self.pragmatic_model(conversation_history = conversation_history, question = question)
        literal_answer_list = self.community_data[community](query_pred.literal_query)
        pragmatic_answer_list = self.community_data[community](query_pred.pragmatic_query)
        return self.response_model(conversation_history = conversation_history, question = question, literal_answers = literal_answer_list, pragmatic_answers = pragmatic_answer_list)


In [126]:
import pprint   
pprint.pprint(pcqa_test[0]['qas'][0])

{'a': 'The Legend of Zelda came out as early as 1986 for the Famicom in Japan, '
      'and was later released in the western world, including Europe and the '
      'US in 1987. Would you like to know about the story?',
 'a_meta': {'literal_obj': [{'endKey': '9cbccabd-66be-4a46-bd8b-f59a299c987d',
                             'startKey': '1f4f808a-8560-4894-b892-15fa3c33887a',
                             'text': 'FDS release February 21, 1986\n'},
                            {'endKey': '738bff65-b4f9-4660-bd18-79722ed67a40',
                             'startKey': 'a0d9d5c5-18bb-4be4-825e-fca2900db18e',
                             'text': 'The Legend of Zelda is the first '
                                     'installment of the Zelda series. '},
                            {'endKey': '738bff65-b4f9-4660-bd18-79722ed67a40',
                             'startKey': '738bff65-b4f9-4660-bd18-79722ed67a40',
                             'text': ' It centers its plot around a boy named 

In [145]:

CM = conversational_model(RAG_prompt, pragmatics_prompt, optimized_response, 3)



In [150]:

CM(question = pcqa_test[0]['qas'][0]['q'], conversation_history = [f"Student: {pcqa_test[0]['qas'][0]['q']}"], community = pcqa_test[0]['community'])

Prediction(
    reasoning="The student's question is about the release year of The Legend of Zelda, based on the conversation history which is just their initial query. The literal_answers provide direct references to the game's release dates, specifically noting the Famicom Disk System release on February 21, 1986, in Japan, which is the original and most relevant fact. The pragmatic_answers offer broader context, like the game's development history and its significance, but I need to prioritize the core fact for accuracy. I'll use the literal answer as the foundation for a straightforward response and incorporate a touch of pragmatic insight to make it engaging, such as mentioning its status as a groundbreaking title. To keep the response natural and human-like, I'll aim for an informal, enthusiastic tone with fluid language, perhaps a minor casual element like a conversational aside, while ensuring it's concise. Finally, I'll end with an open-ended question to encourage more discuss

First Question evaluation

In [155]:
fqe_references = []
fqe_predictions = []

for item in pcqa_test:
    topic = item['topic']
    gernre = item['genre']
    community = item['community']
    qas = item['qas'][0]
    question = qas['q']
    answer = qas['a']
    lit_spans = [l['text'] for l in qas['a_meta']['literal_obj']]
    #lit_answer = ' '.join(lit_spans)
    prag_spans = [l['text'] for l in qas['a_meta']['pragmatic_obj']]
    #prag_answer = ' '.join(prag_spans)
    #rag_context = topic_search[community](question)

    print(question)
    
    fqe_result = CM(question=question, conversation_history = f"Student: {question}", community = community)
    fqe_references.append(dspy.Example(question=question, response=answer).with_inputs("question"))
    fqe_predictions.append(dspy.Prediction(question = question, response = fqe_result.response).with_inputs("question"))
   
    print(f"Answer: '{fqe_result.response}'.")


What year did the Legend of Zelda come out?
Answer: 'Hey, great question! The Legend of Zelda first hit the scene on February 21, 1986, in Japan for the Famicom Disk System—it was a total game-changer that started this epic adventure series. What's your take on it, or maybe you're curious about other Zelda games or its impact on gaming?'.
What console is The Legend of Zelda designed for?
Answer: 'Hey, awesome question! The Legend of Zelda was originally designed for the Famicom back in 1986 in Japan, but it really took off on the NES for folks in the US and Europe. It's this epic adventure that kicked off the whole series and totally changed gaming with its exploration and puzzles—it's been remade and re-released on all sorts of consoles since, which shows how timeless it is. Have you ever played it, or are you curious about checking out some of the newer versions?'.
when did the legend of zelda last until?
Answer: 'Hey, great question about The Legend of Zelda! From what I can tell, t

2025/08/24 19:48:31 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=6000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.1)  if the reason for truncation is repetition.


Answer: 'None'.
Tell me when did the legend of Zelda come out?
Answer: 'Oh, man, The Legend of Zelda is such a classic—it's like the start of something epic! From what I know, it first hit the scene in Japan on February 21, 1986, for the Famicom Disk System, and then made its way to the US on August 22, 1987, for the NES. That game totally revolutionized adventure gaming and kicked off this amazing series. What's got you curious about it—maybe some fun facts about the early games or your favorite part of the story?'.
What system did the game first come out in? 
Answer: 'Oh, hey, great question about The Legend of Zelda! From what I can tell, the games you're thinking of, like The Faces of Evil, first hit the scene on the Philips CD-i back in 1991—it was one of those early multimedia consoles that tried to mix games with other stuff. It's kinda wild how that system played a part in Zelda's history before things really took off on Nintendo's own hardware. What other parts of the Zelda se

2025/08/24 19:48:33 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=6000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.1)  if the reason for truncation is repetition.
2025/08/24 19:48:33 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Answer: 'Oh, Zelda's totally awesome—she's this super wise and brave princess from the Legend of Zelda games, always stepping up to protect Hyrule with her smarts and magical powers! She can do all sorts of cool stuff, like wielding a bow with light arrows to blast away evil, using telepathy to call for help, or even possessing spirits in some adventures to fight alongside Link. Her kindness and self-sacrificing spirit make her a real hero. What's your favorite Zelda game or moment that makes her stand out to you?'.
What is the Legend of Zelda?
Answer: 'Oh, man, The Legend of Zelda is one of the coolest video game series out there! It's all about this epic adventure in a magical world called Hyrule, created by this genius guy named Shigeru Miyamoto back in the '80s. You play as this brave hero named Link, who's on a quest to save Princess Zelda from the big bad Ganon, using stuff like swords, shields, and even magical items to explore dungeons and solve puzzles. It's got this awesome m

...

Answer: 'Oh, hey, if you're asking about what's at the stadium, like Dodger Stadium in LA, it's this iconic spot with amazing views of the city and mountains, hosting epic baseball games and even concerts! It's got that classic vibe with a rich history of stars like Sandy Koufax. What got you thinking about stadiums – are you into sports or maybe planning a visit?'.
What's the highest home runs in a game gotten by the Yankees?


2025/08/24 19:49:18 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=6000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.1)  if the reason for truncation is repetition.


Answer: 'None'.
who is their leader in homeruns?
Answer: 'Oh, hey, that's a great question about baseball! From what I know, the all-time leader in home runs in Major League Baseball is Hank Aaron—he smashed an incredible 755 homers over his career, which is still a huge deal in the sport's history. It's all about that power and legacy, like how home runs can really turn the game around and make legends out of players. Pretty cool, right? What else are you curious about in baseball, or do you have another sport or topic on your mind?'.
what year did the new York win first championship? 
Answer: 'Well, from what I can dig up, the New York Giants snagged their first championship way back in 1888, beating the St. Louis Browns in a series—that's some serious early baseball history! It's cool how New York's teams have been dominating the scene for so long. Are you a big fan of the Giants, or maybe curious about the Yankees' wins too? Let's chat more about it!'.
who founded the new york yank

2025/08/24 19:49:19 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=6000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.1)  if the reason for truncation is repetition.
2025/08/24 19:49:19 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Answer: 'Absolutely, the New York Yankees are based right in New York City, specifically in the Bronx at Yankee Stadium! They've been a huge part of the city's baseball scene for over a century, with all that rich history and those epic rivalries. What else are you curious about when it comes to the Yankees or other sports teams?'.
What sports league does the new York Yankees play in? 
Answer: 'Oh, hey, that's a great question! The New York Yankees play in the American League, specifically the East Division, and they've got this amazing history with a ton of World Series wins that make them one of the legends of baseball. It's all about that intense rivalry and their storied past in the big leagues. Are you a Yankees fan, or what's got you curious about them?'.
who is the all time leader inh rbi's for the yankees?
Answer: 'Alright, let's dive into some Yankee history! The all-time leader in RBIs for the New York Yankees is Lou Gehrig—he racked up an impressive 1,995 RBIs during his car

Note: I've made an error and forgot to set the number of demonstrations.

Since it gives too many, sometimes my response is truncated due to the 6000 token limit.

Let's evaluate:

In [156]:
fqe_examples = []
for pred, reference in zip(fqe_predictions, fqe_references ):
    fqe_examples.append(dspy.Example(example = reference, pred = pred).with_inputs("example" , "pred"))

fqe_score = metric.batch(fqe_examples)

Processed 213 / 213 examples: 100%|██████████| 213/213 [06:17<00:00,  1.77s/it]


In [159]:
total = 0
for s in fqe_score:
    total+=s
print(total / len(fqe_score))

0.3156714382552613


Evaluation:

Compared to the base model, this is an improvement over the answer with retrieved passages. It however is not higher than the answer with literal or pragmatic spans.

My guess is its an issue with precision, where the base model is very percise with the spans while a bit lower on recall. 

The LLM model is probably worse at both since the pragmatic answer might be different.

However as a conversational answer, which isn't scored by the evaluation, it is clearly better.

In [161]:
cost_list = [cur['cost'] for cur in lm.history if cur['cost'] is not None]
total = 0
for c in cost_list:
    total+=c

print(total)

5.661088800000005


Conversation evaluation

In [ ]:
test_list = []

for item in pcqa_test:
    ch = []
    for pair in item['qas']:
        a = pair['a']
        q = pair['q']
        llist = [ans['text'] for ans in pair['a_meta']['literal_obj']]
        plist = [ans['text'] for ans in pair['a_meta']['pragmatic_obj']]
        ch.append(f"Student: {q}")
        current_ch = ch[:]
        test_list.append({
            "topic": item['topic'],
            "community": item['community'],
            "conversation_history": current_ch,
            "literal_answers": llist,
            "pragmatic_answers": plist,
            "question": q,
            "answer": a
            })
        ch.append(f"Teacher: {a}")

test_list_sample = random.sample(test_list, 50)

In [165]:
full_conversation_references = []
full_conversation_predictions = []

for item in test_list_sample:
    topic = item['topic']
    community = item['community']
    conversation_history = item['conversation_history']
    question = item['question']
    answer = item['answer']
    lit_spans = item['literal_answers']
    lit_answer = ' '.join(lit_spans)
    prag_spans = item['pragmatic_answers']
    prag_answer = ' '.join(prag_spans)

    print(question)
    
    result = CM(question=question, conversation_history = conversation_history, community = community)
    full_conversation_references.append(dspy.Example(question=question, response=answer).with_inputs("question"))
    full_conversation_predictions.append(dspy.Prediction(question = question, response = result.response).with_inputs("question"))
   
    print(f"Answer: '{result.response}'.")

Who is Moira?
Answer: 'Oh, hey, Moira Brown is this quirky non-player character from Fallout 3—she's all about that Wasteland Survival Guide thing, and having a high Perception skill lets you chat her up more effectively for some cool tips and quests. She's kinda the go-to gal in the Capital Wasteland for surviving the tough life out there. Pretty fun, right? What other Fallout mysteries are you curious about, or wanna dive deeper into her story?'.
is game in stret gme in my stret 


2025/08/25 13:41:55 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=6000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.1)  if the reason for truncation is repetition.
2025/08/25 13:41:55 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Answer: 'Hey, I think you might mean 'Is the game set in the streets or popular in my street?' For Fallout, yeah, a lot of it takes place in ruined cities and streets after the big nuclear war—like exploring the wastelands of California or D.C., dodging dangers around every corner. It's super immersive! What exactly are you curious about with that, or do you wanna chat more about specific parts of the series?'.
Who voices the robot Gypsy on MST3K?


2025/08/25 13:42:58 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=6000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.1)  if the reason for truncation is repetition.
2025/08/25 13:42:58 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Answer: 'Oh, Gypsy's voice has changed hands over the years, which is pretty cool for a robot character! Originally, she was voiced by Josh Weinstein from 1988 to 1989, then Jim Mallon took over from 1989 to 1997 and again in 2007, followed by Patrick Brantseg from 1997 to 1999. More recently, Rebecca Hanson handled the voice starting in 2017, and Yvonne Freese stepped in around 2019. It's all part of what makes MST3K so timeless. What else are you curious about with the bots or maybe some episodes?'.
What year was the yankees founded? 
Answer: 'Hey, great question about the Yankees! They were founded back in 1901 as the Baltimore Orioles, and then they moved to New York in 1903, which is when they really started building their legendary story. It's wild how that early move set the stage for all their success. What other parts of baseball history or the Yankees would you like to dive into next?'.
What genre is this game? 
Answer: 'Oh, great follow-up! The Legend of Zelda is an action-a

...

Answer: 'Hey, I think you mean 'who selected who as the Dragon Warrior?' In the Kung Fu Panda story, it was the wise old Master Oogway who chose Po for that epic role—kinda like the universe pointing him out during a big tournament! It's all about Oogway's vision and believing in yourself, which makes it super inspiring. What do you wanna know next about Po or the other masters?'.
Hmm, ok I've done these before, usually it doesn't drag so, I'm not paid well to wait so long, but you are.  Which album did Britney put out that was best received. (did it take you a long time to gather information about the last answer, by the way?
Answer: 'Hey, I get it—waiting for answers can feel like forever, but nah, it didn't take me long at all to pull that info together; I'm just here to chat whenever you're ready! So, from what I know, Britney's album "Blackout" from 2007 is often seen as her best received one by critics—it won Album of the Year at the MTV Europe Music Awards and really showcased h

2025/08/25 14:06:08 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=6000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.1)  if the reason for truncation is repetition.
2025/08/25 14:06:08 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Answer: 'Oh, man, The Legend of Zelda is one of the coolest video game series ever! It all kicked off back in 1986 with the first game, created by the legendary Shigeru Miyamoto, where you play as this brave hero named Link on a epic quest to save Princess Zelda from the big bad Ganon in the magical land of Hyrule. It's packed with adventures, tricky dungeons, puzzles, and even a special Second Quest that ramps up the challenge for replay value. What got you interested in it, or have you played any of the games yet?'.
how would you describe her demeanor?


2025/08/25 14:07:27 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=6000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.1)  if the reason for truncation is repetition.
2025/08/25 14:07:27 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Answer: 'Chihiro starts out as this typical scared and kinda whiny kid, you know, with that childlike vibe when she first enters the spirit world. But man, she really grows on you—she becomes super brave, quick-witted, and totally responsible, especially when she's fighting to save her parents and friends. It's like she taps into this inner strength that makes her such a cool heroine! What do you think about how she changes throughout the story?'.
Ok, wow, tell me about Ian, like just a thing or two. Thanks for understanding this is all new to me, I'm thinking this isn't a good game at all.
Answer: 'Hey, no worries at all—I'm glad you're giving this a shot, even if it's feeling a bit overwhelming right now. Ian's a cool guy in Fallout; he's basically a former caravan guard you meet early on in Shady Sands, and he's one of the first buddies you can recruit to join your adventure in the wasteland. He's got that tough, survivalist vibe, which makes him pretty reliable for fighting off mut

In [166]:
full_conversation_examples = []
for pred, reference in zip(full_conversation_predictions, full_conversation_references ):
    full_conversation_examples.append(dspy.Example(example = reference, pred = pred).with_inputs("example" , "pred"))

full_conversation_score = metric.batch(full_conversation_examples)

total = 0
for s in full_conversation_score:
    total+=s
print(total / len(full_conversation_score))

Processed 50 / 50 examples: 100%|██████████| 50/50 [01:25<00:00,  1.72s/it]
0.30374123784888635


In [167]:
cost_list = [cur['cost'] for cur in lm.history if cur['cost'] is not None]
total = 0
for c in cost_list:
    total+=c

print(total)

6.356262200000005


In [172]:

for example, pred, score in zip(test_list_sample, full_conversation_predictions, full_conversation_score):
    pprint.pprint(example['conversation_history'])
    pprint.pprint(example['answer'])
    pprint.pprint(pred.response)
    print(score)

['Student: What is fallout?',
 'Teacher: Fallout is a computer role-playing game, which relies on the '
 'SPECIAL character creation system. Each of the seven letters of SPECIAL '
 'stands for one of the seven character attributes in the game.',
 'Student: Cool what does the S stand for please?',
 'Teacher: Strength is the attribute for this letter, which affects two '
 'important game properties: carry weight and the minimum strength for '
 'weapons.',
 'Student: What weapons can you use in the game?',
 'Teacher: A character can use many weapons such as energy weapons, big guns '
 'and small guns. The players can also rely on their Agility tokens during '
 'fights when they have an Agility-based weapon.',
 'Student: Great thanks. What does the T stand for ?',
 'Teacher: I do not know, but the seven SPECIAL attributes are Strength, '
 'Perception, Endurance, Charisma, Intelligence, Agility and Luck.',
 'Student: Ah yes sorry. What does the perception ability let you do?',
 'Teacher: Pe

Since some of the answers with a low (0) score were also answers that were low rated by students, lets test on a currated list

In [ ]:


currated_test_list = []

for item in pcqa_test:
    ch = []
    for pair in item['qas']:
        a = pair['a']
        q = pair['q']
        llist = [ans['text'] for ans in pair['a_meta']['literal_obj']]
        plist = [ans['text'] for ans in pair['a_meta']['pragmatic_obj']]
        ch.append(f"Student: {q}")
        if "human_eval" in pair:
            avg = 0
            for score in pair['human_eval']:
                avg+=int(score)
            avg/=5
            if avg >=4:
                current_ch = ch[:]
                currated_test_list.append({
                    "topic": item['topic'],
                    "community": item['community'],
                    "conversation_history": current_ch,
                    "literal_answers": llist,
                    "pragmatic_answers": plist,
                    "question": q,
                    "answer": a,
                    "score": avg
                    })
        ch.append(f"Teacher: {a}")

print(len(currated_test_list))


56


In [ ]:

currated_fc_references = []
currated_fc_predictions = []

for item in currated_test_list:
    topic = item['topic']
    community = item['community']
    conversation_history = item['conversation_history']
    question = item['question']
    answer = item['answer']
    lit_spans = item['literal_answers']
    lit_answer = ' '.join(lit_spans)
    prag_spans = item['pragmatic_answers']
    prag_answer = ' '.join(prag_spans)
    
    result = CM(question=question, conversation_history = conversation_history, community = community)
    currated_fc_references.append(dspy.Example(question=question, response=answer).with_inputs("question"))
    currated_fc_predictions.append(dspy.Prediction(question = question, response = result.response).with_inputs("question"))

    



2025/08/25 18:31:26 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=6000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.1)  if the reason for truncation is repetition.
2025/08/25 18:31:26 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/08/25 18:36:30 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=6000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.1)  if the reason for truncation is repetition.
2025/08/25 18:36:30 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/08/25

In [175]:

currated_fc_examples = []

for pred, reference in zip(currated_fc_predictions, currated_fc_references ):
    currated_fc_examples.append(dspy.Example(example = reference, pred = pred).with_inputs("example" , "pred"))

currated_fc_score = metric.batch(currated_fc_examples)

total = 0
for s in currated_fc_score:
    total+=s
print(total / len(currated_fc_score))


Processed 56 / 56 examples: 100%|██████████| 56/56 [01:51<00:00,  2.00s/it]
0.16538523019248275


Let's look over the results

In [178]:
print(f"{len(currated_test_list)} , {len(currated_fc_predictions)}")
for example, pred, score in zip(currated_test_list, currated_fc_predictions, currated_fc_score):
    pprint.pprint(example['conversation_history'])
    pprint.pprint(example['answer'])
    pprint.pprint(example['score'])
    pprint.pprint(pred.response)
    print(score)

56 , 56
['Student: Will LEGO have any new themes for 2023?']
"I don't know"
5.0
("Oh, man, I'm always excited about new LEGO themes—it's like they're "
 'constantly surprising us with fresh ideas! From what I can tell based on '
 "LEGO's history of rolling out awesome stuff like Star Wars sets and their "
 'big events, they definitely keep things innovative and might have some cool '
 "surprises lined up for 2023. But hey, specifics can change, so it's best to "
 'watch for official announcements. What kind of themes are you hoping to see, '
 'like something adventurous or maybe space-themed?')
0.0
['Student: what is the latest most popular lego set?',
 'Teacher: One of the most successful LEGO sets was teh LEGO train system '
 'first released in 1966, which included a four and a half Volt motor and '
 'rails that was later expanded to a twelve Volt motor two years later.',
 'Student: was it powered by batteries or did you plug it in?',
 'Teacher: The first LEGO train sets used battery

I see a glaring issue with some of them, have a look:

In [ ]:
for item in currated_test_list:
  if item['question'] == "What kinds of movies?": # a question about MST3K
    community = item['community']
    conversation_history = item['conversation_history']
    question = item['question']
    answer = item['answer']
    
    result = CM(question=question, conversation_history = conversation_history, community = community)
    pprint.pprint(result)

Prediction(
    reasoning='The student\'s question, "Ah sorry, who is your favourite character?", builds directly on the conversation history, which has been a deep dive into the series involving characters like Gilgamesh, Sparks, and Agatha. This is a light-hearted follow-up, so I need to keep the response fun and supportive to maintain the engaging teacher-student dynamic. The literal_answers specify "Agatha Heterodyne 1 through 5," which I interpret as Agatha Heterodyne being the favorite character, likely referencing her prominence in the early parts of the series. This provides the core fact for my answer. The pragmatic_answers add depth by noting that "Agatha\'s appearance is based on the way Phil used to draw Kaja Foglio," which I can use to make the response more interesting and contextual without straying from the provided info. In my reasoning, I\'ll ensure the reply is accurate, natural, and informal—perhaps with a casual tone, like using contractions or a playful vibe to mi

In [ ]:
print(currated_fc_predictions[56]) #an answer about unrelated fandom

Prediction(
    question='What kinds of movies?',
    response="Oh, hey, no worries at all—great question! My favorite character in the series has gotta be Agatha Heterodyne; she's just so awesome across those first few installments, like 1 through 5, bringing all that wild adventure and cleverness to the story. And fun fact, her look is actually based on how Phil used to draw Kaja Foglio, which makes her even more intriguing, right? Who's your pick for a favorite character, or is there something else about the series you're curious about?"
)


In [183]:
dspy.inspect_history()





[2025-08-26T15:21:49.498830]

System message:

Your input fields are:
1. `conversation_history` (list[str]): 
2. `question` (str): 
3. `literal_answers` (list[str]): 
4. `pragmatic_answers` (list[str]):
Your output fields are:
1. `reasoning` (str): 
2. `response` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## conversation_history ## ]]
{conversation_history}

[[ ## question ## ]]
{question}

[[ ## literal_answers ## ]]
{literal_answers}

[[ ## pragmatic_answers ## ]]
{pragmatic_answers}

[[ ## reasoning ## ]]
{reasoning}

[[ ## response ## ]]
{response}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an engaging and knowledgeable teacher in a conversational AI system designed to discuss popular culture topics such as literature, films, video games, music, sports, and celebrities. Your task is to handle interactions based on the provided 'conversation_history'—a list of prior ex

It seems in this example the LLM confused a few shot demonstration's input with the actual input.

From looking over the whole prompt, I don't see an issue - the demonstrations end and then the input is given.

I could redo the optimmization without demonstrations, or let go of optimizing entirely.

Also, this curration picked up a lot of examples from one MST3K conversation where the teacher's answers are frequently unrelated to the questions, and the student still rated them highly.

I'll use the normal sample for evaluation.

In [185]:
for item, pred, score in zip(test_list_sample, full_conversation_predictions, full_conversation_score):
    pprint.pprint(item['conversation_history'])
    pprint.pprint(item['answer'])
    pprint.pprint(pred.response)
    print(score)

['Student: What is fallout?',
 'Teacher: Fallout is a computer role-playing game, which relies on the '
 'SPECIAL character creation system. Each of the seven letters of SPECIAL '
 'stands for one of the seven character attributes in the game.',
 'Student: Cool what does the S stand for please?',
 'Teacher: Strength is the attribute for this letter, which affects two '
 'important game properties: carry weight and the minimum strength for '
 'weapons.',
 'Student: What weapons can you use in the game?',
 'Teacher: A character can use many weapons such as energy weapons, big guns '
 'and small guns. The players can also rely on their Agility tokens during '
 'fights when they have an Agility-based weapon.',
 'Student: Great thanks. What does the T stand for ?',
 'Teacher: I do not know, but the seven SPECIAL attributes are Strength, '
 'Perception, Endurance, Charisma, Intelligence, Agility and Luck.',
 'Student: Ah yes sorry. What does the perception ability let you do?',
 'Teacher: Pe


Evaluation:

- Comparison of Models: How did the performance of the "traditional" QA model compare to the LLM-based model? 

Answer: Looking over the responses of the traditional model and the LLM model, it is clearly the LLM model that is better in every way.

The answers are conversational, they convey the percise literal answer just as often if not more frequently than the base model, and they produce a pragmatic answer as wel.

One weakness of the answers is that they are often too long, and I feel they are too friendly - they seem dishonest.

------------------------------------------------------------------------------------------------------------------------------------------

- Theory of Mind: 

This is the model's greatest failure in my opinion.

A lot of its pragmatic additions tend to misunderstand what the student is really interested at.

Either adding too much pragmatic information or failing to understand the pragmatic interest.

------------------------------------------------------------------------------------------------------------------------------------------

- Here is one good example:

 'Student: What made this funny, were the effects convincing? (Or not??).

'Teacher: This episode was interesting because it was made over new year's and there "
 'was a countdown to 1989."

LLM: 'Oh, man, Gamera vs. Zigra is a classic MST3K gem because the humor comes '
 
 "from how ridiculously over-the-top everything is—the crew's witty riffs "
 
 'totally poke fun at the not-so-convincing effects, like that giant turtle '
 
 "battling a shark monster in the cheesiest ways possible. It's all about "

 'those clever jokes and host segments that make the bad stuff hilariously '

'entertaining. What do you think made it funny, or do you want to chat about '

 'another MST3K movie next?')

 The LLM failed to mention how this specific episode is unique among the rest due to the countdown the teacher mentioned.

 It likely just copied key information from the episode synopsis.

------------------------------------------------------------------------------------------------------------------------------------------

In general the main reason the LLM god a low score is precision - gave too many details that were not in the original teacher's answer.

Other cases where the LLM scored a 0 that aren't related are:

- The original teacher's answer was a variation of "I don't know"

- The LLM answered a few-shot example's input instead of the correct one. This happened about once in 20 example (very roughly)

- The LLM response was "None", probably related to a mistake I made where the prompt was over 6,000 tokens long due to too many demonstrations.

------------------------------------------------------------------------------------------------------------------------------------------

- Here is a hard example:

"Student: Hmm, ok I've done these before, usually it doesn't drag so, I'm not "

 'paid well to wait so long, but you are.  Which album did Britney put out '
 
 'that was best received. (did it take you a long time to gather information '
 
 'about the last answer, by the way?']

Teacher: 'Baby One More Time was named the greatest debut single of all time in 2020. '
 
 'It also held a 15 year record for selling 1.3 million copies in the first '
 
 'week.')

LLM: ("Hey, I get it—waiting for answers can feel like forever, but nah, it didn't "
 
 "take me long at all to pull that info together; I'm just here to chat "
 
 'whenever you\'re ready! So, from what I know, Britney\'s album "Blackout" '
 
 'from 2007 is often seen as her best received one by critics—it won Album of '
 
 'the Year at the MTV Europe Music Awards and really showcased her growth with '
 
 "that edgy, dance-heavy vibe. It's got that raw energy that fans and "
 
 'reviewers loved. What else are you curious about with Britney, or maybe '
 
 'something from her early days that caught your eye?')


 The LLM's answer may have been more accurate to the student's literal use of "album"

 but failed to understand the student's interest in which release catapolted Britney to popularity, which happens to be a single.

------------------------------------------------------------------------------------------------------------------------------------------

 - 'Student: Pearl was a great actor right?',
 
  'Teacher: Yes, I think so. She is played by Mary Jo Pehl, an American actress '
 
 'and writer. She is best known for her portrayal of the sinister Mrs. Pearl '
 
 'Forrester, the mother of Dr. Clayton Forrester on Mystery Science Theater '
 
 '3000.',

 'Student: oh wow.. that is amazing']

Teacher: 'Yes. She also served as a member of the writing staff on MST3K since 1992.'

('Oh yeah, Mary Jo Pehl is totally amazing—she was born in 1960 in Minnesota '
 
 'and kicked off her comedy career after ditching nursing, which is hilarious '
 
 "since she hated blood! She's not just known for playing Pearl Forrester on "

 "MST3K; she's also got her own Twitch show called Movie Jo Night where she "

 'watches and riffs on movies live. Pretty cool, right? What else about her or '
 
 'the show catches your interest?')

 This example and others like it are interesting, since the student is leading the teacher into elaborating on their previous answer.

 The result is that the teacher could go into what personally interests them, which is hard to predict pragmatically.

 I don't find the LLM's answer necessarily worse here, the score was 0.

------------------------------------------------------------------------------------------------------------------------------------------

Here is an example with a potential issue:

 'Teacher: My apologies. There is a diverse selection of various recruitable '
 
 'non-player characters that can be found to aid the player character in the '
 
 'post-apocalyptic wasteland. One such example is Ian, who is a former caravan '
 
 'guard that can be found in Shady Sands and is the first recruitable '
 
 'non-player character that the player character typically meets.',
 
 'Student: Ok, wow, tell me about Ian, like just a thing or two. Thanks for '
 
 "understanding this is all new to me, I'm thinking this isn't a good game at "

 'all.']

Teacher: 'He is an experienced traveler and gunman that can equip any pistol or 10mm '
 
 'SMG (including "large handguns"), and wears a leather jacket. Unlike in '
 
 'Fallout 2,, there is no limit to the number of non-player characters that '
 
 'may be recruited in Fallout.')

LLM: "Hey, no worries at all—I'm glad you're giving this a shot, even if it's "
 
 "feeling a bit overwhelming right now. Ian's a cool guy in Fallout; he's "
 
 "basically a former caravan guard you meet early on in Shady Sands, and he's "
 
 'one of the first buddies you can recruit to join your adventure in the '
 
 "wasteland. He's got that tough, survivalist vibe, which makes him pretty "
 
 'reliable for fighting off mutants and stuff. Fallout can be intense at '
 
 "first, but it gets really fun once you get into the groove—trust me, it's "
 
 'got some epic stories! What do you think, wanna hear about another character '
 
 "or maybe why the game's worth another try?")

 In this example the LLM repeated information from past teacher's answers, which is obviously terrible.

 It might be worth adding a "do not repeat info already discussed" in the prompt.

------------------------------------------------------------------------------------------------------------------------------------------

All in all, aside from pragmatical differences and technical issues, the responses got a really high score, regularly over 0.4 and somtimes even reaching a 0.8